# Adding Memory to Agents

### Working with Pre-requisites to configure the Agent Properly

In [8]:
# Import core dependencies to create the agent, for Agent Framework
import asyncio
import os
import json

from dotenv import load_dotenv, find_dotenv
# Core components for building Agent, tool-enabled agents
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient
from agent_framework import AgentThread

## Load Environment Variables

Load environment variables from a `.env` file in the project directory. This file should contain:
- `OPENROUTER_ENDPOINT`: The OpenRouter API endpoint URL
- `OPENROUTER_API_KEY`: Your OpenRouter API key

In [9]:
# load environment file
load_dotenv(find_dotenv())

True

## Setup Chat Client

Configure the `OpenAIChatClient` to use OpenRouter API, which provides access to various LLM models including NVIDIA's Nemotron model. The client is configured with:

- `base_url`: The OpenRouter API endpoint
- `api_key`: Your API key for authentication
- `model_id`: The specific model to use (NVIDIA Nemotron 3 Nano 30B in this case)

In [26]:
# Setup OpenAIChatClient for LLM Inference - Here we will use OpenRouter API which is compatible with OpenAI and NVIDIA 30B model
# This client connects to the OpenRouter Models which are OpenAI-compatible endpoint
# Environment variables required
# OPENROUTER_ENDPOINT - 
# OPENROUTER_API_KEY
openai_chat_client = OpenAIChatClient(
    base_url=os.environ.get("GROQ_ENDPOINT"),
    api_key=os.environ.get("GROQ_API_KEY"),
    model_id="openai/gpt-oss-120b"
)

In [27]:
AGENT_NAME = "FoodAgent"

AGENT_INSTRUCTIONS = """You are an expert AI Chef dedicated to helping users discover and prepare delicious meals. Keep it concise, short and effective.
"""

## Create the Food Agent

Create the first agent (`food_agent`) with:

- `name`: "FoodAgent"
- `chat_client`: The OpenAI chat client configured earlier
- `instructions`: The behavior instructions defined above
- `chat_message_store`: External Database Store for persistent message.

In [28]:
# Create the Memory Class to hold the memory
# Adding structured Output here
from pydantic import BaseModel
from typing import Annotated, List, Dict, Any, Optional

class MealInfo(BaseModel):
    """Information about a Meal"""
    id: str | None = None
    name: str | None = None
    category: str | None = None
    area: str | None = None
    instructions: str | None = None
    ingredients: List[str] | None = None
    tags: str | None = None
    youtube_link: str | None = None

The FoodInfoMemory class contains the following behavior:

    Extraction: It uses a chat client to actively scan user messages for details regarding the MealInfo schema (specifically looking for the Name, Category, Area, or Ingredients) whenever new messages are added to the thread at the end of a run.

    Context Injection: It injects the current known state of the MealInfo object into the system prompt before every agent invocation so the agent knows what food is being discussed.

    Gatekeeping: If the meal_info.name is missing, it explicitly instructs the agent to stop and ask the user for the name of the meal, prohibiting the agent from hallucinating details or answering complex questions until the basic identity of the meal is established.

    Serialization: It implements serialization (to_dict / from_dict) to allow the complex MealInfo object to be persisted as part of the thread state, ensuring the agent "remembers" the ingredients or instructions across different chat sessions.

In [33]:
import json
from typing import Any, List, Sequence, MutableSequence
from pydantic import BaseModel
from agent_framework import ContextProvider, ChatMessage, Context, ChatClientProtocol

# Your Pydantic Model
class MealInfo(BaseModel):
    id: str | None = None
    name: str | None = None
    category: str | None = None
    area: str | None = None
    instructions: str | None = None
    ingredients: List[str] | None = None
    tags: str | None = None
    youtube_link: str | None = None

class FoodInfoMemory(ContextProvider):
    def __init__(self, chat_client: ChatClientProtocol, meal_info: MealInfo | None = None, **kwargs: Any):
        self._chat_client = chat_client
        if meal_info:
            self.meal_info = meal_info
        elif kwargs:
            self.meal_info = MealInfo.model_validate(kwargs)
        else:
            self.meal_info = MealInfo()

    async def invoked(
        self,
        request_messages: ChatMessage | Sequence[ChatMessage],
        response_messages: ChatMessage | Sequence[ChatMessage] | None = None,
        invoke_exception: Exception | None = None,
        **kwargs: Any,
    ) -> None:
        
        user_messages = [msg for msg in request_messages if hasattr(msg, "role") and msg.role == "user"]
        
        if user_messages:
            print(f"DEBUG: Attempting extraction on user message: '{user_messages[-1].content}'")
            try:
                # 1. Ask LLM for extraction
                result = await self._chat_client.get_response(
                    messages=request_messages,
                    instructions=(
                        "Extract the meal details (name, category, ingredients, instructions, etc.) "
                        "from the conversation into JSON format."
                    ),
                    options={"response_format": MealInfo}, 
                )

                extracted_data = None

                # 2. Try to get structured value directly
                if hasattr(result, 'value') and result.value:
                    print("DEBUG: Successfully received structured 'value' from client.")
                    extracted_data = result.value
                
                # 3. Fallback: Parse raw content if 'value' is missing (Common fix)
                elif hasattr(result, 'content') and result.content:
                    raw_text = result.content
                    print(f"DEBUG: Structured 'value' was None. Trying to parse raw content: {raw_text}")
                    
                    # Clean markdown code blocks if present
                    if "```" in raw_text:
                        raw_text = raw_text.split("```json")[-1].split("```")[0].strip()
                        if raw_text.startswith("json"): # Handle cases like ```json ...
                             raw_text = raw_text[4:]
                    
                    try:
                        data_dict = json.loads(raw_text)
                        extracted_data = MealInfo(**data_dict)
                        print("DEBUG: Successfully parsed raw JSON content.")
                    except json.JSONDecodeError:
                        print("DEBUG: ERROR - Raw content was not valid JSON.")

                # 4. Update Memory
                if extracted_data:
                    self._merge_info(extracted_data)
                    print(f"DEBUG: Memory Updated. Current Name: {self.meal_info.name}")
                else:
                    print("DEBUG: WARNING - No data could be extracted from the LLM response.")

            except Exception as e:
                # This will print the actual error to your notebook
                print(f"DEBUG: CRITICAL ERROR in 'invoked': {e}")
                import traceback
                traceback.print_exc()

    def _merge_info(self, extracted: MealInfo):
        """Helper to merge non-null fields"""
        if self.meal_info.name is None and extracted.name:
            self.meal_info.name = extracted.name
        if self.meal_info.category is None and extracted.category:
            self.meal_info.category = extracted.category
        if self.meal_info.area is None and extracted.area:
            self.meal_info.area = extracted.area
        if self.meal_info.instructions is None and extracted.instructions:
            self.meal_info.instructions = extracted.instructions
        if self.meal_info.ingredients is None and extracted.ingredients:
            self.meal_info.ingredients = extracted.ingredients
        if self.meal_info.youtube_link is None and extracted.youtube_link:
            self.meal_info.youtube_link = extracted.youtube_link

    async def invoking(self, messages: ChatMessage | MutableSequence[ChatMessage], **kwargs: Any) -> Context:
        instructions: list[str] = []
        if self.meal_info.name is None:
            instructions.append(
                "Gatekeeper: You do NOT know the meal name. Ask the user for the name of the meal."
            )
        else:
            instructions.append(f"Context: The user is discussing '{self.meal_info.name}'.")
            if self.meal_info.ingredients:
                instructions.append(f"Ingredients: {', '.join(self.meal_info.ingredients)}.")
        return Context(instructions=" ".join(instructions))

    def serialize(self) -> str:
        return self.meal_info.model_dump_json()

In [34]:
memory_provider = FoodInfoMemory(openai_chat_client)
agent = ChatAgent(
    name = AGENT_NAME,
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
    context_provider=memory_provider
)

In [35]:
thread = agent.get_new_thread()

In [36]:
# --- CELL: Initialization ---

# 1. Initialize Memory
memory_provider = FoodInfoMemory(openai_chat_client)

# 2. Initialize Agent
# Note: Instructions are generic; the Memory Provider will inject the specific meal context.
agent = ChatAgent(
    chat_client=openai_chat_client,
    instructions="You are a helpful cooking assistant.", 
    context_provider=memory_provider
)

# 3. Create Thread
thread = agent.get_new_thread()

# --- CELL: Execution Test ---

print("--- Turn 1: Asking for ingredients without giving a name ---")
# Expectation: Agent should refuse and ask for the name (Gatekeeper logic)
response = await agent.run("What are the ingredients?", thread=thread)
print(f"Agent: {response.text}")

print("\n--- Turn 2: Providing the Name ---")
# Expectation: Agent should acknowledge the name. Memory extracts "Spaghetti Carbonara".
response = await agent.run("I want to make Spaghetti Carbonara", thread=thread)
print(f"Agent: {response.text}")

print("\n--- Turn 3: Providing more details ---")
# Expectation: Memory extracts ingredients into the list.
response = await agent.run("It needs eggs, cheese, and guanciale.", thread=thread)
print(f"Agent: {response.text}")

# --- CELL: Inspection ---
# --- CELL: Inspection ---

# Verify what is actually stored in the memory object
# Since you passed a single provider, thread.context_provider IS your memory object.
current_memory = thread.context_provider

# Check if it is indeed your class
if isinstance(current_memory, FoodInfoMemory):
    print(f"\n[MEMORY DUMP]")
    # Use model_dump_json for a pretty print, or access fields directly
    print(current_memory.meal_info.model_dump_json(indent=2))
    
    # Or individual fields:
    # print(f"Meal Name: {current_memory.meal_info.name}")
    # print(f"Ingredients: {current_memory.meal_info.ingredients}")
else:
    print("Context provider is not FoodInfoMemory or is None")

if isinstance(current_memory, FoodInfoMemory):
    print(f"\n[MEMORY DUMP]")
    print(f"Meal Name: {current_memory.meal_info.name}")
    print(f"Ingredients: {current_memory.meal_info.ingredients}")

--- Turn 1: Asking for ingredients without giving a name ---
Agent: Sure! Could you let me know which meal you’d like the ingredient list for?

--- Turn 2: Providing the Name ---
Agent: ### Spaghetti Carbonara – Ingredient List (Serves 4)

| Ingredient | Amount | Notes |
|------------|--------|-------|
| Spaghetti | 400 g (14 oz) | Use high‑quality durum wheat pasta |
| Pancetta (or guanciale) | 150 g (5 oz) | Cut into small dice; guanciale gives a richer flavor |
| Large eggs | 3 (plus 1 egg‑yolk) | At room temperature |
| Pecorino Romano cheese | 100 g (1 cup) | Finely grated; you can blend half Pecorino, half Parmesan if you prefer |
| Freshly ground black pepper | 1–2 tsp | Coarse grind for texture and aroma |
| Kosher salt | for pasta water | About 1 Tbsp per 4 L of water |
| Optional: garlic clove | 1, lightly crushed | Remove before adding pancetta if you like a subtle hint |
| Optional: fresh parsley | 1 Tbsp, chopped | For garnish |

#### Quick Tips
- **Egg‑cheese mixture:** I